# Answer Quality Filter: Training and Evaluation

**Objective:** Train a DeBERTa-v3-small binary classifier on labeled_asqa.csv
to accept correct answers and reject hallucinated ones, then evaluate against
a no-filter baseline on a held-out test set.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import sys, json, logging
import numpy as np
import pandas as pd

from rag_filtering.filtering.data_split import load_and_split
from rag_filtering.filtering.learned_filter import AnswerQualityClassifier, train_classifier
from rag_filtering.filtering.filter_evaluator import FilterEvaluator

logging.basicConfig(level=logging.INFO)
SEED = 42

c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\venv\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
RAGAS not installed. Install with: pip install ragas


## 1. Data Split

In [2]:
train_df, val_df, test_df = load_and_split("../data/asqa/labeled_asqa.csv", seed=SEED)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Label distribution (train): {train_df['label'].value_counts().to_dict()}")

INFO:src.filtering.data_split:Loaded 8706 samples from ../data/asqa/labeled_asqa.csv
INFO:src.filtering.data_split:Found 4353 unique base question IDs
INFO:src.filtering.data_split:Train: 5570 samples (pos=2785, neg=2785)
INFO:src.filtering.data_split:Val: 1394 samples (pos=697, neg=697)
INFO:src.filtering.data_split:Test: 1742 samples (pos=871, neg=871)


Train: 5570, Val: 1394, Test: 1742
Label distribution (train): {1: 2785, 0: 2785}


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from rag_filtering.filtering.learned_filter import _extract_top1_context

for df in [train_df, val_df, test_df]:
    df["context_text"] = df["context"].apply(_extract_top1_context)

print(f"Context extracted. Sample:")
print(f"  Q: {train_df['question'].iloc[0][:80]}")
print(f"  C: {train_df['context_text'].iloc[0][:80]}")
print(f"  A: {train_df['answer'].iloc[0][:80]}")
print(f"  L: {train_df['label'].iloc[0]}")

In [ ]:
# Step 1: Confirm label mapping.
# CONVENTION: label==1 means correct/faithful, label==0 means hallucinated.
# Use a known pair (asqa_X, asqa_Xb) as visual proof.
print("=== STEP 1: LABEL MAPPING ASSERTION ===")

_known_base = None
for _sid in train_df["id"].tolist():
    if _sid.endswith("b"):
        continue
    if (train_df["id"] == _sid + "b").any():
        _known_base = _sid
        break

assert _known_base is not None, "No paired (asqa_X, asqa_Xb) found in train_df"

_pos_row = train_df[train_df["id"] == _known_base].iloc[0]
_neg_row = train_df[train_df["id"] == _known_base + "b"].iloc[0]
print(f"Pair: {_known_base} vs {_known_base}b\n")
print(f"  POSITIVE (label={_pos_row['label']}, must be 1):")
print(f"    A: {_pos_row['answer'][:160]}...")
print(f"  NEGATIVE (label={_neg_row['label']}, must be 0):")
print(f"    A: {_neg_row['answer'][:160]}...")

assert int(_pos_row["label"]) == 1, (
    f"Expected base id label=1, got {_pos_row['label']}"
)
assert int(_neg_row["label"]) == 0, (
    f"Expected 'b' id label=0, got {_neg_row['label']}"
)
print("\nLabel mapping VERIFIED: 1 = correct, 0 = hallucinated")


In [ ]:
# Step 2: Inspect 5 paired (correct, hallucinated, context) examples.
# Manually verify hallucinations are subtle entity swaps, NOT random noise.
print("=== STEP 2: INSPECT 5 PAIRED EXAMPLES ===\n")

inspected = 0
_seen_bases = set()
for _sid in train_df["id"].tolist():
    _base = _sid[:-1] if _sid.endswith("b") else _sid
    if _base in _seen_bases:
        continue
    _pm = train_df[train_df["id"] == _base]
    _nm = train_df[train_df["id"] == _base + "b"]
    if len(_pm) == 0 or len(_nm) == 0:
        continue
    _p = _pm.iloc[0]
    _n = _nm.iloc[0]
    _seen_bases.add(_base)
    inspected += 1
    print(f"--- Pair {inspected} ({_base}) ---")
    print(f"  Q:    {_p['question'][:140]}")
    print(f"  CTX:  {_p['context_text'][:220]}...")
    print(f"  +A:   {_p['answer'][:160]}")
    print(f"  -A:   {_n['answer'][:160]}")
    print()
    if inspected >= 5:
        break

print("Manual check: hallucinations should differ from +A in entities only.")
print("If both look identical or both look totally different, data is broken.")


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# Step 5 + Step 6: Overfit sanity GATE.
# A 184M-parameter model MUST overfit 32 paired examples to train_F1 >= 0.95.
# If it cannot, the code/data/format is broken — DO NOT scale up to full training.
from rag_filtering.filtering.learned_filter import overfit_sanity_check

print("=== STEP 5+6: OVERFIT SANITY GATE ===")
overfit_result = overfit_sanity_check(train_df, n_pairs=16, epochs=50)
print(f"\nOverfit result: {overfit_result}")

assert overfit_result["train_f1"] >= 0.95, (
    f"OVERFIT FAILED — train F1={overfit_result['train_f1']:.3f} < 0.95. "
    "The code/data/format is broken; the 5 worst-classified pairs were "
    "logged above. DO NOT run full training until this passes."
)
print("\nOVERFIT PASSED — safe to proceed to full training.")


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# Step 12 (pre-training baseline): 3-class NLI zero-shot.
# Cache the result so we can compare against the fine-tuned model AFTER training.
# If the raw MNLI head beats the fine-tuned one on FPR, fine-tuning is hurting.
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rag_filtering.filtering.filter_evaluator import select_threshold_min_fpr

print("=== STEP 12 (pre-training): 3-CLASS NLI ZERO-SHOT BASELINE ===")
_NLI3C = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
_nli_tok = AutoTokenizer.from_pretrained(_NLI3C)
_nli_m = AutoModelForSequenceClassification.from_pretrained(_NLI3C)
_nli_m.to("cuda" if torch.cuda.is_available() else "cpu").eval()
_ent_idx = next(
    i for i, n in _nli_m.config.id2label.items() if n.lower() == "entailment"
)
_device = next(_nli_m.parameters()).device


def _nli3c_entailment_probs(contexts, answers, batch_size=32, max_length=384):
    out = []
    for s in range(0, len(contexts), batch_size):
        bc = [f"Premise: {c}" for c in contexts[s:s + batch_size]]
        ba = [f"Hypothesis: {a}" for a in answers[s:s + batch_size]]
        enc = _nli_tok(
            bc, ba, return_tensors="pt", truncation=True,
            padding=True, max_length=max_length,
        ).to(_device)
        with torch.no_grad():
            lg = _nli_m(**enc).logits
        out.extend(torch.softmax(lg, dim=-1)[:, _ent_idx].cpu().tolist())
    return out


_nli3c_test_conf = _nli3c_entailment_probs(
    test_df["context_text"].tolist(), test_df["answer"].tolist(),
)
_nli3c_test_labels = test_df["label"].tolist()

# Default (untuned) threshold = 0.5
_def_preds = [c >= 0.5 for c in _nli3c_test_conf]
_def_tp = sum(p and l == 1 for p, l in zip(_def_preds, _nli3c_test_labels))
_def_tn = sum((not p) and l == 0 for p, l in zip(_def_preds, _nli3c_test_labels))
_def_fp = sum(p and l == 0 for p, l in zip(_def_preds, _nli3c_test_labels))
_def_fn = sum((not p) and l == 1 for p, l in zip(_def_preds, _nli3c_test_labels))
nli3c_baseline_default = {
    "threshold": 0.5,
    "fpr": _def_fp / (_def_fp + _def_tn) if (_def_fp + _def_tn) > 0 else 0.0,
    "recall": _def_tp / (_def_tp + _def_fn) if (_def_tp + _def_fn) > 0 else 0.0,
    "precision": _def_tp / (_def_tp + _def_fp) if (_def_tp + _def_fp) > 0 else 0.0,
    "f1": (
        2 * _def_tp / (2 * _def_tp + _def_fp + _def_fn)
        if (2 * _def_tp + _def_fp + _def_fn) > 0 else 0.0
    ),
}
print(f"  zero-shot @ t=0.5: {nli3c_baseline_default}")

nli3c_baseline = select_threshold_min_fpr(
    _nli3c_test_conf, _nli3c_test_labels, min_recall=0.70,
)
print(
    f"  tuned (FP-min @ R>=0.70): t={nli3c_baseline['threshold']:.3f}, "
    f"FPR={nli3c_baseline['fpr']:.3f}, R={nli3c_baseline['recall']:.3f}, "
    f"P={nli3c_baseline['precision']:.3f}, F1={nli3c_baseline['f1']:.3f}"
)

del _nli_m, _nli_tok
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("3-class NLI baseline cached; GPU freed.")


## 2. Train Classifier

In [ ]:
import shutil
from pathlib import Path

model_dir = Path("../models/answer_filter")
if model_dir.exists():
    shutil.rmtree(model_dir)
    print(f"Removed old model at {model_dir}, retraining...")

model_dir = train_classifier(
    train_df, val_df,
    output_dir=str(model_dir),
)

INFO:src.filtering.learned_filter:Extracting top-1 context for training (dropout=0.15)


Removed old model at ..\models\answer_filter, retraining...


INFO:src.filtering.learned_filter:Context extracted: 5570/5570 train samples have non-empty context
INFO:src.filtering.learned_filter:DIAGNOSTIC [train]: labels={1: 2785, 0: 2785}, NaN(label=0, q=0, a=0), label_dtype=int64
INFO:src.filtering.learned_filter:DIAGNOSTIC [val]: labels={1: 697, 0: 697}, NaN(label=0, q=0, a=0), label_dtype=int64
INFO:src.filtering.learned_filter:Training config: {
  "model_name": "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
  "model_path": "models/answer_filter",
  "threshold": 0.5,
  "max_length": 384,
  "batch_size": 16,
  "num_epochs": 10,
  "learning_rate": 2e-05,
  "classifier_lr": 0.001,
  "warmup_ratio": 0.1,
  "weight_decay": 0.01,
  "max_grad_norm": 1.0,
  "label_smoothing": 0.0,
  "early_stopping_patience": 3,
  "fp16": false,
  "save_total_limit": 3,
  "seed": 42,
  "use_context": true,
  "context_dropout": 0.15
}
INFO:src.filtering.learned_filter:Model: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli  ->  ..\models\answer_filter
INFO:src.filterin

Epoch,Training Loss,Validation Loss


## 3. Threshold Tuning (Validation Set)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# Step 9 + Step 10: FP-minimization threshold sweep (replaces max-F1 sweep).
# Selects the threshold with the lowest FPR subject to recall >= min_recall.
# F1 is intentionally NOT the selection criterion — it can mask high FPR.
from rag_filtering.filtering.filter_evaluator import select_threshold_min_fpr
import yaml as _yaml

clf = AnswerQualityClassifier(str(model_dir))
evaluator = FilterEvaluator()

val_decisions = clf.predict_batch(
    val_df["context_text"].tolist(), val_df["answer"].tolist()
)
val_confidences = [d.confidence for d in val_decisions]
val_labels = val_df["label"].tolist()

with open("../configs/filtering/deberta_filter.yaml") as fh:
    _cfg = _yaml.safe_load(fh)["learned_filter"]
min_recall_floor = _cfg.get("min_recall_for_threshold", 0.70)

selection = select_threshold_min_fpr(
    val_confidences, val_labels, min_recall=min_recall_floor,
)
best_threshold = selection["threshold"]
clf.threshold = best_threshold   # apply tuned operating point to the classifier
sweep_df = pd.DataFrame(selection["sweep"]).round(4)

from pathlib import Path as _Path
import json as _json
_Path("../results").mkdir(parents=True, exist_ok=True)
with open("../results/threshold_selection.json", "w") as fh:
    _json.dump(
        {k: v for k, v in selection.items() if k != "sweep"},
        fh, indent=2,
    )

print(
    f"Selected threshold (min FPR @ recall>={min_recall_floor:.2f}): "
    f"t={best_threshold:.3f} | FPR={selection['fpr']:.3f} "
    f"R={selection['recall']:.3f} P={selection['precision']:.3f} "
    f"F1={selection['f1']:.3f}"
)
sweep_df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep_df["threshold"], sweep_df["precision"], label="Precision", marker="o")
ax.plot(sweep_df["threshold"], sweep_df["recall"], label="Recall", marker="s")
ax.plot(sweep_df["threshold"], sweep_df["f1"], label="F1", marker="^", linewidth=2)
ax.axvline(best_threshold, color="gray", linestyle="--", label=f"Best t={best_threshold}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold Tuning (Validation Set)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Final Evaluation (Test Set)

In [ ]:
test_decisions = clf.predict_batch(test_df["context_text"].tolist(), test_df["answer"].tolist())
test_preds = [d.confidence >= best_threshold for d in test_decisions]
test_labels = test_df["label"].tolist()

learned_result = evaluator.evaluate(test_preds, test_labels)
baseline_result = evaluator.compute_no_filter_baseline(test_labels)

comparison = evaluator.compare(
    {"No Filter": baseline_result, "Learned Filter": learned_result},
    save_path="../results/filter_comparison.json",
)
pd.DataFrame(comparison).set_index("strategy").round(4)

In [ ]:
# Step 7: 2x2 confusion matrix + FPR + ROC-AUC on the test set.
# FPR is the thesis north-star metric.
from sklearn.metrics import confusion_matrix, roc_auc_score

test_conf = [d.confidence for d in test_decisions]
cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])
tn, fp, fn, tp = (int(cm[0, 0]), int(cm[0, 1]),
                  int(cm[1, 0]), int(cm[1, 1]))
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
try:
    roc_auc = float(roc_auc_score(test_labels, test_conf))
except ValueError:
    roc_auc = float("nan")

print("=== STEP 7: CONFUSION MATRIX + FPR (test set) ===")
print(f"Threshold used: {best_threshold:.3f}\n")
print(f"                  pred=reject   pred=accept")
print(f"  label=halluc:   TN={tn:5d}      FP={fp:5d}")
print(f"  label=correct:  FN={fn:5d}      TP={tp:5d}")
print()
print(f"FPR (FP / (FP+TN)):       {fpr:.4f}   <- PRIMARY thesis metric")
print(f"TPR / Recall:             {tpr:.4f}")
print(f"ROC-AUC (rank quality):   {roc_auc:.4f}")
print()
print("Goal: FPR -> 0 while keeping recall >= 0.70.")


## 5. Error Analysis

In [ ]:
test_df = test_df.copy()
test_df["predicted"] = test_preds
test_df["confidence"] = [d.confidence for d in test_decisions]

fp = test_df[(test_df["label"] == 0) & (test_df["predicted"] == True)]
fn = test_df[(test_df["label"] == 1) & (test_df["predicted"] == False)]
print(f"False Positives (hallucinations accepted): {len(fp)}")
print(f"False Negatives (correct answers rejected): {len(fn)}")
print(f"\nSample False Positives:")
for _, row in fp.head(3).iterrows():
    print(f"  [{row['id']}] conf={row['confidence']:.3f}: {row['answer'][:120]}...")
print(f"\nSample False Negatives:")
for _, row in fn.head(3).iterrows():
    print(f"  [{row['id']}] conf={row['confidence']:.3f}: {row['answer'][:120]}...")

## 6. Apply to Real RAG Outputs

In [ ]:
import ast
import pandas as pd

rag_df = pd.read_csv("../results/asqa_normal_rag_predictions.csv")

def _top1_passage(contexts_str, max_chars=800):
    """contexts column is a stringified list of passage strings; take top-1."""
    try:
        passages = ast.literal_eval(contexts_str)
        if isinstance(passages, (list, tuple)) and passages:
            return str(passages[0])[:max_chars]
    except (ValueError, SyntaxError):
        pass
    return str(contexts_str)[:max_chars]

rag_df["context_text"] = rag_df["contexts"].apply(_top1_passage)

clf.threshold = best_threshold  # 0.570
rag_decisions = clf.predict_batch(
    rag_df["context_text"].tolist(),
    rag_df["predicted_answer"].tolist(),
)
rag_df["filter_accept"]     = [d.accept for d in rag_decisions]
rag_df["filter_confidence"] = [round(d.confidence, 4) for d in rag_decisions]

n_accept = int(rag_df["filter_accept"].sum())
print(f"Accepted: {n_accept}/{len(rag_df)} ({100*n_accept/len(rag_df):.1f}%)")

cols = ["id", "question", "predicted_answer", "context_text",
        "filter_confidence", "filter_accept"]

print("\n=== Most-confidently REJECTED (check: are these really unfaithful?) ===")
display(rag_df.sort_values("filter_confidence").head(15)[cols])

print("\n=== Borderline (confidence near 0.570) ===")
rag_df["_dist"] = (rag_df["filter_confidence"] - 0.570).abs()
display(rag_df.sort_values("_dist").head(15)[cols])

rag_df.drop(columns="_dist").to_csv(
    "../results/rag_predictions_filtered.csv", index=False)

## 7. Ablation: Training Data Size

Train on 25%, 50%, 75%, 100% of the data to measure how much labelled data the filter needs.

In [ ]:
fractions = [0.25, 0.50, 0.75, 1.0]
data_size_rows = []

for frac in fractions:
    tag = f"data_{int(frac * 100)}pct"
    n_samples = int(len(train_df) * frac)
    subset = train_df.sample(n=n_samples, random_state=SEED).reset_index(drop=True)
    out_dir = Path(f"../models/ablations/{tag}")

    if out_dir.exists() and (out_dir / "config.json").exists():
        print(f"  {tag}: model exists, skipping training.")
    else:
        print(f"  Training {tag} ({n_samples} samples)...")
        train_classifier(subset, val_df, output_dir=str(out_dir))

    abl_clf = AnswerQualityClassifier(str(out_dir))
    # find best threshold on val
    abl_decisions = abl_clf.predict_batch(val_df["context_text"].tolist(), val_df["answer"].tolist())
    abl_conf = [d.confidence for d in abl_decisions]
    best_abl_f1, best_abl_t = 0.0, 0.5
    for t in np.arange(0.1, 0.95, 0.05):
        preds = [c >= t for c in abl_conf]
        r = evaluator.evaluate(preds, val_df["label"].tolist())
        if r.f1 > best_abl_f1:
            best_abl_f1 = r.f1
            best_abl_t = round(float(t), 2)
    # evaluate on test
    test_dec = abl_clf.predict_batch(test_df["context_text"].tolist(), test_df["answer"].tolist())
    test_p = [d.confidence >= best_abl_t for d in test_dec]
    r = evaluator.evaluate(test_p, test_df["label"].tolist())
    data_size_rows.append({"fraction": frac, "n_train": n_samples, "threshold": best_abl_t,
                           "f1": r.f1, "precision": r.precision, "recall": r.recall, "accuracy": r.accuracy})
    print(f"  {tag}: F1={r.f1:.4f} Acc={r.accuracy:.4f} (t={best_abl_t})")

ds_df = pd.DataFrame(data_size_rows)
ds_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ds_df["fraction"] * 100, ds_df["f1"], marker="o", label="F1", linewidth=2)
ax.plot(ds_df["fraction"] * 100, ds_df["accuracy"], marker="s", label="Accuracy")
ax.set_xlabel("Training Data (%)")
ax.set_ylabel("Score")
ax.set_title("Ablation: Training Data Size")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Ablation: Max Sequence Length

Compare max_length 256, 384, 512 to measure the trade-off between speed and accuracy.

In [ ]:
max_lengths = [256, 384, 512]
ml_rows = []

for ml in max_lengths:
    tag = f"maxlen_{ml}"
    out_dir = Path(f"../models/ablations/{tag}")

    if out_dir.exists() and (out_dir / "config.json").exists():
        print(f"  {tag}: model exists, skipping training.")
    else:
        print(f"  Training {tag}...")
        train_classifier(
            train_df, val_df,
            output_dir=str(out_dir),
            config_overrides={"max_length": ml},
        )

    abl_clf = AnswerQualityClassifier(str(out_dir))
    abl_clf.max_length = ml
    # find best threshold on val
    abl_decisions = abl_clf.predict_batch(val_df["context_text"].tolist(), val_df["answer"].tolist())
    abl_conf = [d.confidence for d in abl_decisions]
    best_abl_f1, best_abl_t = 0.0, 0.5
    for t in np.arange(0.1, 0.95, 0.05):
        preds = [c >= t for c in abl_conf]
        r = evaluator.evaluate(preds, val_df["label"].tolist())
        if r.f1 > best_abl_f1:
            best_abl_f1 = r.f1
            best_abl_t = round(float(t), 2)
    # evaluate on test
    test_dec = abl_clf.predict_batch(test_df["context_text"].tolist(), test_df["answer"].tolist())
    test_p = [d.confidence >= best_abl_t for d in test_dec]
    r = evaluator.evaluate(test_p, test_df["label"].tolist())
    ml_rows.append({"max_length": ml, "threshold": best_abl_t,
                    "f1": r.f1, "precision": r.precision, "recall": r.recall, "accuracy": r.accuracy})
    print(f"  {tag}: F1={r.f1:.4f} Acc={r.accuracy:.4f} (t={best_abl_t})")

ml_df = pd.DataFrame(ml_rows)
ml_df

## 9. Ablation: Model Size

Compare DeBERTa-v3-small (44M), base (86M), and large (304M) to measure
the effect of parametric knowledge on closed-book answer verification.

In [ ]:
model_variants = [
    ("microsoft/deberta-v3-small", "deberta_small"),
    ("microsoft/deberta-v3-base", "deberta_base"),
    ("microsoft/deberta-v3-large", "deberta_large"),
]
model_rows = []

for model_id, tag in model_variants:
    out_dir = Path(f"../models/ablations/{tag}")

    if out_dir.exists() and (out_dir / "config.json").exists():
        print(f"  {tag}: model exists, skipping training.")
    else:
        print(f"  Training {tag} ({model_id})...")
        train_classifier(
            train_df, val_df,
            output_dir=str(out_dir),
            config_overrides={"model_name": model_id},
        )

    abl_clf = AnswerQualityClassifier(str(out_dir))
    abl_decisions = abl_clf.predict_batch(
        val_df["context_text"].tolist(), val_df["answer"].tolist()
    )
    abl_conf = [d.confidence for d in abl_decisions]
    best_abl_f1, best_abl_t = 0.0, 0.5
    for t in np.arange(0.1, 0.95, 0.05):
        preds = [c >= t for c in abl_conf]
        r = evaluator.evaluate(preds, val_df["label"].tolist())
        if r.f1 > best_abl_f1:
            best_abl_f1 = r.f1
            best_abl_t = round(float(t), 2)

    test_dec = abl_clf.predict_batch(
        test_df["context_text"].tolist(), test_df["answer"].tolist()
    )
    test_p = [d.confidence >= best_abl_t for d in test_dec]
    r = evaluator.evaluate(test_p, test_df["label"].tolist())
    model_rows.append({
        "model": tag, "threshold": best_abl_t,
        "f1": r.f1, "precision": r.precision, "recall": r.recall,
        "accuracy": r.accuracy, "rejection_recall": r.rejection_recall,
        "rejection_rate": r.rejection_rate,
    })
    print(f"  {tag}: F1={r.f1:.4f} Acc={r.accuracy:.4f} (t={best_abl_t})")

model_df = pd.DataFrame(model_rows)
model_df

## 10. NLI Zero-Shot Filter (no training)

Use a pre-trained NLI model as a zero-shot answer quality filter.
Frames the task as: "Does the question entail the answer?"

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from rag_filtering.filtering.nli_filter import NLIAnswerFilter

nli_filter = NLIAnswerFilter()

nli_val_decisions = nli_filter.predict_batch(
    val_df["context_text"].tolist(), val_df["answer"].tolist()
)
nli_val_conf = [d.confidence for d in nli_val_decisions]
nli_val_labels = val_df["label"].tolist()

nli_sweep = []
for t in np.arange(0.1, 0.95, 0.05):
    preds = [c >= t for c in nli_val_conf]
    r = evaluator.evaluate(preds, nli_val_labels)
    nli_sweep.append({
        "threshold": round(float(t), 2),
        "f1": r.f1, "precision": r.precision, "recall": r.recall,
        "accuracy": r.accuracy,
        "rejection_recall": r.rejection_recall,
        "rejection_rate": r.rejection_rate,
    })

nli_sweep_df = pd.DataFrame(nli_sweep)
nli_best_row = nli_sweep_df.loc[nli_sweep_df["f1"].idxmax()]
nli_best_threshold = nli_best_row["threshold"]
print(f"NLI Best threshold: {nli_best_threshold} (F1={nli_best_row['f1']:.4f})")

nli_test_decisions = nli_filter.predict_batch(
    test_df["context_text"].tolist(), test_df["answer"].tolist()
)
nli_test_preds = [d.confidence >= nli_best_threshold for d in nli_test_decisions]
nli_result = evaluator.evaluate(nli_test_preds, test_df["label"].tolist())

nli_comparison = evaluator.compare(
    {
        "No Filter": baseline_result,
        "Learned Filter": learned_result,
        "NLI Zero-Shot": nli_result,
    },
    save_path="../results/filter_comparison_with_nli.json",
)
pd.DataFrame(nli_comparison).set_index("strategy").round(4)

## 11. Ensemble Filter (DeBERTa + NLI + Lexical features)

Combine the learned DeBERTa confidence, NLI entailment score, and
lexical features (token overlap, answer length) into a logistic
regression meta-classifier.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from rag_filtering.filtering.ensemble_filter import EnsembleFilter

ensemble = EnsembleFilter(
    deberta_clf=clf,
    nli_filter=nli_filter,
)

train_metrics = ensemble.fit(
    train_df["context_text"].tolist(),
    train_df["answer"].tolist(),
    train_df["label"].tolist(),
)
print(f"Ensemble train metrics: {train_metrics}")

ensemble_val_decisions = ensemble.predict_batch(
    val_df["context_text"].tolist(), val_df["answer"].tolist()
)
ensemble_val_conf = [d.confidence for d in ensemble_val_decisions]

ens_sweep = []
for t in np.arange(0.1, 0.95, 0.05):
    preds = [c >= t for c in ensemble_val_conf]
    r = evaluator.evaluate(preds, val_df["label"].tolist())
    ens_sweep.append({
        "threshold": round(float(t), 2),
        "f1": r.f1, "precision": r.precision, "recall": r.recall,
        "accuracy": r.accuracy,
    })

ens_sweep_df = pd.DataFrame(ens_sweep)
ens_best_row = ens_sweep_df.loc[ens_sweep_df["f1"].idxmax()]
ens_best_threshold = ens_best_row["threshold"]
print(f"Ensemble best threshold: {ens_best_threshold} (F1={ens_best_row['f1']:.4f})")

ensemble_test_decisions = ensemble.predict_batch(
    test_df["context_text"].tolist(), test_df["answer"].tolist()
)
ens_test_preds = [d.confidence >= ens_best_threshold for d in ensemble_test_decisions]
ensemble_result = evaluator.evaluate(ens_test_preds, test_df["label"].tolist())

ensemble.save("../models/ensemble_filter")

all_comparison = evaluator.compare(
    {
        "No Filter": baseline_result,
        "Learned Filter": learned_result,
        "NLI Zero-Shot": nli_result,
        "Ensemble": ensemble_result,
    },
    save_path="../results/filter_comparison_all.json",
)
pd.DataFrame(all_comparison).set_index("strategy").round(4)

## 12. Results Summary

In [ ]:
print("=== FINAL RESULTS (all 6 required metrics) ===")
print(f"Trained on: {len(train_df)} samples, Evaluated on: {len(test_df)} samples")

strategies = [
    ("No Filter", baseline_result, "-"),
    ("Learned (small)", learned_result, best_threshold),
    ("NLI Zero-Shot", nli_result, nli_best_threshold),
    ("Ensemble", ensemble_result, ens_best_threshold),
]

header = (
    f"{'Strategy':<18} {'t':>5} {'P':>6} {'R':>6} {'F1':>6} "
    f"{'Acc':>6} {'RejR':>6} {'RejRate':>8}"
)
print(f"\n{header}")
print("-" * len(header))
for name, r, t in strategies:
    t_str = f"{t}" if isinstance(t, str) else f"{t:.2f}"
    print(
        f"{name:<18} {t_str:>5} {r.precision:>6.3f} {r.recall:>6.3f} "
        f"{r.f1:>6.3f} {r.accuracy:>6.3f} "
        f"{r.rejection_recall:>6.3f} {r.rejection_rate:>8.3f}"
    )

print(f"\nRAG outputs: {n_accept}/{len(rag_df)} accepted ({100*n_accept/len(rag_df):.1f}%)")

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# Step 12 (post-training): side-by-side comparison with 3-class NLI baseline.
# If the raw MNLI head (zero-shot, no training) beats our fine-tuned binary
# model on FPR, our fine-tuning is hurting and we should reconsider.
from rag_filtering.filtering.filter_evaluator import select_threshold_min_fpr

print("=== STEP 12: SIDE-BY-SIDE: LEARNED BINARY vs 3-CLASS NLI ZERO-SHOT ===\n")

learned_row = {
    "model": "Learned binary (fine-tuned)",
    "threshold": float(best_threshold),
    "fpr": learned_result.fp / (learned_result.fp + learned_result.tn)
        if (learned_result.fp + learned_result.tn) > 0 else 0.0,
    "recall": learned_result.recall,
    "precision": learned_result.precision,
    "f1": learned_result.f1,
}

nli3c_zero_row = {
    "model": "NLI 3-class zero-shot @ t=0.5",
    "threshold": 0.5,
    "fpr": nli3c_baseline_default["fpr"],
    "recall": nli3c_baseline_default["recall"],
    "precision": nli3c_baseline_default["precision"],
    "f1": nli3c_baseline_default["f1"],
}

nli3c_tuned_row = {
    "model": "NLI 3-class + FP-min threshold",
    "threshold": nli3c_baseline["threshold"],
    "fpr": nli3c_baseline["fpr"],
    "recall": nli3c_baseline["recall"],
    "precision": nli3c_baseline["precision"],
    "f1": nli3c_baseline["f1"],
}

cmp_df = pd.DataFrame([learned_row, nli3c_zero_row, nli3c_tuned_row]).round(4)
print(cmp_df.to_string(index=False))
print("\nThesis target: lowest FPR while recall >= 0.70.")
print("If the zero-shot 3-class NLI beats the learned model on FPR, the fine-tune is hurting.")

import json as _json
with open("../results/step12_3class_nli_comparison.json", "w") as fh:
    _json.dump(cmp_df.to_dict(orient="records"), fh, indent=2)
print("\nSaved -> ../results/step12_3class_nli_comparison.json")


## 13. External Benchmarks (WikiEval + MS MARCO)

Cross-domain generalization test: the filter is trained on **ASQA**, then
evaluated unchanged on two external labeled faithfulness datasets. Each CSV
already uses the filter's native schema — `context` (premise), `answer`
(hypothesis), `label` (1 = faithful, 0 = hallucinated).

Threshold-independent **ROC-AUC** is the fair generalization metric; the
operating-point metrics (FPR/Recall/F1) use the ASQA-tuned `best_threshold`,
so they may be slightly off-calibrated on a new domain.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def evaluate_external(csv_path, name, threshold):
    """Run the trained DeBERTa filter on an external labeled CSV.

    Expects columns: context (premise), answer (hypothesis), label (1=faithful).
    """
    df = pd.read_csv(csv_path).dropna(subset=["context", "answer", "label"]).reset_index(drop=True)
    df["label"] = df["label"].astype(int)

    clf.threshold = threshold
    decisions = clf.predict_batch(df["context"].tolist(), df["answer"].tolist())
    conf = [d.confidence for d in decisions]
    preds = [c >= threshold for c in conf]

    res = evaluator.evaluate(preds, df["label"].tolist())

    y, p = np.array(df["label"]), np.array(preds, dtype=int)
    fp = int(((p == 1) & (y == 0)).sum())
    tn = int(((p == 0) & (y == 0)).sum())
    fpr = fp / (fp + tn) if (fp + tn) else float("nan")
    auc = roc_auc_score(y, conf) if len(np.unique(y)) > 1 else float("nan")

    print(f"\n=== {name} (n={len(df)}, pos={int(y.sum())}, neg={int((y == 0).sum())}) @ t={threshold:.3f} ===")
    print(f"  Precision={res.precision:.3f}  Recall={res.recall:.3f}  F1={res.f1:.3f}")
    print(f"  Accuracy ={res.accuracy:.3f}  FPR={fpr:.3f}  ROC-AUC={auc:.3f}  <- AUC is the cross-domain metric")

    df["filter_confidence"] = conf
    df["filter_accept"] = preds
    return df, {
        "dataset": name, "n": len(df), "precision": res.precision,
        "recall": res.recall, "f1": res.f1, "accuracy": res.accuracy,
        "fpr": fpr, "roc_auc": auc, "threshold": threshold,
    }


external_rows = []
wikieval_df, wikieval_metrics = evaluate_external(
    "../data/wikiEval/labeled_wikieval.csv", "WikiEval", best_threshold)
external_rows.append(wikieval_metrics)
msmarco_df, msmarco_metrics = evaluate_external(
    "../data/ms-macro/labeled_msmacro.csv", "MS MARCO", best_threshold)
external_rows.append(msmarco_metrics)

external_df = pd.DataFrame(external_rows)
external_df.to_csv("../results/external_benchmark_results.csv", index=False)
external_df